## 0 · GPU Verification

In [1]:
import subprocess, torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

n_gpus = torch.cuda.device_count()
print(f"PyTorch {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU count      : {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}  VRAM={props.total_memory/1e9:.1f} GB")

DEVICE = (
    ",".join(str(i) for i in range(n_gpus))
    if n_gpus > 1
    else ("0" if n_gpus == 1 else "cpu")
)
print(f'\nUsing device: "{DEVICE}"')

Sat Mar 28 08:04:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1 · Environment Setup

In [2]:
import sys, subprocess

for pkg in ["ultralytics>=8.2.0"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"])

import os, time, yaml, json, zipfile, shutil
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from rich.console import Console
from rich.table import Table
from IPython.display import display, FileLink
from ultralytics import YOLO
import ultralytics

console = Console()
print(f"ultralytics {ultralytics.__version__} ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics 8.4.30 ready


## 2 · Config

In [3]:
# ── Source data (read-only Kaggle input) ──────────────────────────────────────
DATA_ROOT = Path(
    "/kaggle/input/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow/css-data"
)

# ── Working directories ────────────────────────────────────────────────────────
WORK_DIR = Path("/kaggle/working")
SPLIT_DIR = WORK_DIR / "split_data"  # re-split symlink tree lives here
OUTPUT_DIR = WORK_DIR / "outputs"
YAML_PATH = WORK_DIR / "ppe_data.yaml"

OUTPUT_DIR.mkdir(exist_ok=True)

CLASS_NAMES = [
    "Hardhat",
    "Mask",
    "NO-Hardhat",
    "NO-Mask",
    "NO-Safety Vest",
    "Person",
    "Safety Cone",
    "Safety Vest",
    "machinery",
    "vehicle",
]
NC = len(CLASS_NAMES)

SEED = 42

# Split ratios (must sum to 1)
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

# ── Hyperparameters (tuned for 2×T4, max mAP, <1hr total) ────────────────────
HP = dict(
    model_size="m",  # yolov8m: best mAP/speed on this dataset
    epochs=100,
    imgsz=640,
    batch=32,  # 16 per GPU on 2×T4
    lr0=0.01,
    lrf=0.01,  # cosine decay floor = lr0 * lrf = 1e-4
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    cos_lr=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    close_mosaic=10,
    label_smoothing=0.1,
    patience=30,
    workers=4,
    cache=True,  # RAM cache; T4 nodes have ~30 GB RAM
    amp=True,
    seed=SEED,
)

# RUN_NAME is fixed so TRAIN_DIR can be set deterministically before training
RUN_NAME = f"yolov8{HP['model_size']}_{HP['epochs']}e"
TRAIN_DIR = WORK_DIR / "runs" / "detect" / RUN_NAME

t = Table(title="Hyperparameters")
t.add_column("Param")
t.add_column("Value")
for k, v in HP.items():
    t.add_row(k, str(v))
console.print(t)
print(f"Run name : {RUN_NAME}")
print(f"Train dir: {TRAIN_DIR}")

      Hyperparameters       
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Param           ┃ Value  ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ model_size      │ m      │
│ epochs          │ 100    │
│ imgsz           │ 640    │
│ batch           │ 32     │
│ lr0             │ 0.01   │
│ lrf             │ 0.01   │
│ momentum        │ 0.937  │
│ weight_decay    │ 0.0005 │
│ warmup_epochs   │ 3.0    │
│ cos_lr          │ True   │
│ hsv_h           │ 0.015  │
│ hsv_s           │ 0.7    │
│ hsv_v           │ 0.4    │
│ degrees         │ 5.0    │
│ translate       │ 0.1    │
│ scale           │ 0.5    │
│ shear           │ 2.0    │
│ fliplr          │ 0.5    │
│ mosaic          │ 1.0    │
│ mixup           │ 0.1    │
│ copy_paste      │ 0.1    │
│ close_mosaic    │ 10     │
│ label_smoothing │ 0.1    │
│ patience        │ 30     │
│ workers         │ 4      │
│ cache           │ True   │
│ amp             │ True   │
│ seed            │ 42     │
└─────────────────┴────────┘

Run name : yolov8m_100e
Train dir: /kaggle/working/runs/detect/yolov8m_100e


## 3 · Merge All Splits & Stratified Re-split

In [4]:
# ── 1. Collect every (image_path, label_path) across all original splits ──────
SRC_SPLITS = [
    (DATA_ROOT / "train" / "images", DATA_ROOT / "train" / "labels"),
    (DATA_ROOT / "valid" / "images", DATA_ROOT / "valid" / "labels"),
    (DATA_ROOT / "test" / "images", DATA_ROOT / "test" / "labels"),
]

all_imgs, all_lbls = [], []
for img_dir, lbl_dir in SRC_SPLITS:
    for img_path in sorted(img_dir.glob("*.jpg")):
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        all_imgs.append(img_path)
        all_lbls.append(lbl_path if lbl_path.exists() else None)

print(f"Total images found: {len(all_imgs)}")

Total images found: 2801


In [5]:
# ── 2. Dominant-class label per image (used as stratification key) ─────────────
def dominant_class(lbl_path):
    """Returns the class_id with the most annotations in the label file.
    Returns NC (a synthetic 'background' bucket) if file is missing or empty."""
    if lbl_path is None or not lbl_path.exists():
        return NC
    counts = Counter()
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                counts[int(parts[0])] += 1
    return counts.most_common(1)[0][0] if counts else NC


strat_labels = [
    dominant_class(lbl) for lbl in tqdm(all_lbls, desc="Building stratification labels")
]

# Show per-class image counts before splitting
label_counts = Counter(strat_labels)
print("\nDominant-class distribution (image count):")
for cls_id, cnt in sorted(label_counts.items()):
    name = CLASS_NAMES[cls_id] if cls_id < NC else "background"
    print(f"  {cls_id:2d}  {name:<18}  {cnt}")

Building stratification labels:   0%|          | 0/2801 [00:00<?, ?it/s]


Dominant-class distribution (image count):
   0  Hardhat             80
   1  Mask                79
   2  NO-Hardhat          31
   3  NO-Mask             48
   4  NO-Safety Vest      133
   5  Person              1286
   6  Safety Cone         238
   7  Safety Vest         61
   8  machinery           730
   9  vehicle             91
  10  background          24


In [6]:
# ── 3. Stratified 80 / 10 / 10 split ──────────────────────────────────────────
# sklearn requires every class to have at least 2 samples in the split;
# merge very rare buckets into the 'background' bucket to avoid that constraint.
MIN_SAMPLES = 5
clean_labels = [lbl if label_counts[lbl] >= MIN_SAMPLES else NC for lbl in strat_labels]

indices = list(range(len(all_imgs)))

train_idx, tmp_idx = train_test_split(
    indices,
    test_size=VAL_RATIO + TEST_RATIO,
    stratify=[clean_labels[i] for i in indices],
    random_state=SEED,
)

val_ratio_of_tmp = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
val_idx, test_idx = train_test_split(
    tmp_idx,
    test_size=1 - val_ratio_of_tmp,
    stratify=[clean_labels[i] for i in tmp_idx],
    random_state=SEED,
)

print(f"Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")
print(
    f"Ratios — Train: {len(train_idx)/len(indices):.2f}  "
    f"Val: {len(val_idx)/len(indices):.2f}  "
    f"Test: {len(test_idx)/len(indices):.2f}"
)

Train: 2240  Val: 280  Test: 281
Ratios — Train: 0.80  Val: 0.10  Test: 0.10


In [7]:
# ── 4. Build symlink tree under SPLIT_DIR ─────────────────────────────────────
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

for split_name in ("train", "val", "test"):
    (SPLIT_DIR / split_name / "images").mkdir(parents=True)
    (SPLIT_DIR / split_name / "labels").mkdir(parents=True)


def make_split(name, idx_list):
    img_dir = SPLIT_DIR / name / "images"
    lbl_dir = SPLIT_DIR / name / "labels"
    for i in tqdm(idx_list, desc=f"Linking {name}", leave=False):
        img_src = all_imgs[i]
        lbl_src = all_lbls[i]
        os.symlink(img_src, img_dir / img_src.name)
        if lbl_src and lbl_src.exists():
            os.symlink(lbl_src, lbl_dir / lbl_src.name)
        else:
            # create empty label file for images with no annotations
            (lbl_dir / (img_src.stem + ".txt")).touch()


make_split("train", train_idx)
make_split("val", val_idx)
make_split("test", test_idx)

print(f"Symlink tree ready at {SPLIT_DIR}")

Linking train:   0%|          | 0/2240 [00:00<?, ?it/s]

Linking val:   0%|          | 0/280 [00:00<?, ?it/s]

Linking test:   0%|          | 0/281 [00:00<?, ?it/s]

Symlink tree ready at /kaggle/working/split_data


## 4 · Write data.yaml

In [8]:
data_yaml = dict(
    path=str(SPLIT_DIR),
    train="train/images",
    val="val/images",
    test="test/images",
    nc=NC,
    names=CLASS_NAMES,
)

with open(YAML_PATH, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print("data.yaml written")
console.print_json(json.dumps(data_yaml))

data.yaml written


{
  "path": "/kaggle/working/split_data",
  "train": "train/images",
  "val": "val/images",
  "test": "test/images",
  "nc": 10,
  "names": [
    "Hardhat",
    "Mask",
    "NO-Hardhat",
    "NO-Mask",
    "NO-Safety Vest",
    "Person",
    "Safety Cone",
    "Safety Vest",
    "machinery",
    "vehicle"
  ]
}

## 5 · Dataset Stats

In [9]:
def annotation_counts(lbl_dir):
    counts = Counter()
    for lf in lbl_dir.glob("*.txt"):
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counts[int(parts[0])] += 1
    return counts


print("Counting annotations per split...")
tr_c = annotation_counts(SPLIT_DIR / "train" / "labels")
vl_c = annotation_counts(SPLIT_DIR / "val" / "labels")
te_c = annotation_counts(SPLIT_DIR / "test" / "labels")

df = pd.DataFrame(
    {
        "class": CLASS_NAMES,
        "train": [tr_c.get(i, 0) for i in range(NC)],
        "val": [vl_c.get(i, 0) for i in range(NC)],
        "test": [te_c.get(i, 0) for i in range(NC)],
    }
)
df["total"] = df[["train", "val", "test"]].sum(axis=1)

print(
    f"\nImages  — train: {len(train_idx)}  val: {len(val_idx)}  test: {len(test_idx)}"
)
print(
    f"Annotations — train: {sum(tr_c.values())}  val: {sum(vl_c.values())}  test: {sum(te_c.values())}"
)
print()
console.print(df.sort_values("total", ascending=False).to_string(index=False))

Counting annotations per split...

Images  — train: 2240  val: 280  test: 281
Annotations — train: 30684  val: 3554  test: 4114



class  train  val  test  total
        Person   7908  923  1041   9872
     machinery   4274  505   567   5346
NO-Safety Vest   3269  422   467   4158
   Safety Cone   2782  337   383   3502
       Hardhat   2667  276   391   3334
       NO-Mask   2634  282   334   3250
   Safety Vest   2556  266   313   3135
    NO-Hardhat   1943  225   259   2427
          Mask   1346  168   186   1700
       vehicle   1305  150   173   1628

## 6 · Training

In [10]:
print("=" * 60)
print(f'Model   : YOLOv8{HP["model_size"]}')
print(f"Device  : {DEVICE}")
print(f'Epochs  : {HP["epochs"]}  Batch: {HP["batch"]}  Imgsz: {HP["imgsz"]}')
print(f"Run dir : {TRAIN_DIR}")
print("=" * 60)

model = YOLO(f"yolov8{HP['model_size']}.pt")

T0 = time.time()

model.train(
    data=str(YAML_PATH),
    epochs=HP["epochs"],
    imgsz=HP["imgsz"],
    batch=HP["batch"],
    device=DEVICE,
    lr0=HP["lr0"],
    lrf=HP["lrf"],
    momentum=HP["momentum"],
    weight_decay=HP["weight_decay"],
    warmup_epochs=HP["warmup_epochs"],
    cos_lr=HP["cos_lr"],
    hsv_h=HP["hsv_h"],
    hsv_s=HP["hsv_s"],
    hsv_v=HP["hsv_v"],
    degrees=HP["degrees"],
    translate=HP["translate"],
    scale=HP["scale"],
    shear=HP["shear"],
    fliplr=HP["fliplr"],
    mosaic=HP["mosaic"],
    mixup=HP["mixup"],
    copy_paste=HP["copy_paste"],
    close_mosaic=HP["close_mosaic"],
    label_smoothing=HP["label_smoothing"],
    patience=HP["patience"],
    workers=HP["workers"],
    cache=HP["cache"],
    amp=HP["amp"],
    seed=HP["seed"],
    project=str(WORK_DIR / "runs" / "detect"),
    name=RUN_NAME,
    exist_ok=True,
    plots=False,  # disabled — saves time and removes overhead
    verbose=True,
)

train_time = time.time() - T0
print(f"\nTraining complete — {train_time/60:.1f} min")
print(f'Best weights: {TRAIN_DIR / "weights" / "best.pt"}')

Model   : YOLOv8m
Device  : 0,1
Epochs  : 100  Batch: 32  Imgsz: 640
Run dir : /kaggle/working/runs/detect/yolov8m_100e
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.30 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/ppe_data.yaml, degrees=5.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, k

## 7 · Training Log

In [11]:
results_csv = TRAIN_DIR / "results.csv"
df_res = pd.read_csv(results_csv)
df_res.columns = df_res.columns.str.strip()

# Print last 10 epochs and best epoch
best_idx = df_res["metrics/mAP50(B)"].idxmax()
best_ep = int(df_res.loc[best_idx, "epoch"])
best_map50 = df_res.loc[best_idx, "metrics/mAP50(B)"]

print(f"Best epoch    : {best_ep}  (mAP@0.5 = {best_map50:.4f})")
print(f'Stopped at    : epoch {int(df_res["epoch"].iloc[-1])}')
print()
print("Last 10 epochs:")
cols = [
    "epoch",
    "train/box_loss",
    "train/cls_loss",
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
]
avail = [c for c in cols if c in df_res.columns]
console.print(df_res[avail].tail(10).to_string(index=False))

Best epoch    : 88  (mAP@0.5 = 0.8786)
Stopped at    : epoch 100

Last 10 epochs:


epoch  train/box_loss  train/cls_loss  metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  
metrics/mAP50-95(B)
    91         0.62364         0.38344               0.92280            0.81080           0.87019              
0.64214
    92         0.61177         0.36301               0.91430            0.81210           0.87236              
0.64445
    93         0.60089         0.36008               0.91996            0.80955           0.87315              
0.64398
    94         0.59129         0.34839               0.92413            0.80846           0.87320              
0.64677
    95         0.59797         0.35180               0.91176            0.81560           0.87330              
0.64825
    96         0.59767         0.34709               0.91310            0.81724           0.87446              
0.64882
    97         0.58723         0.34890               0.91588            0.81622           0.87398              
0.64946
    98         0.58658         0.34364               0.91738            0.81168           0.87440              
0.64960
    99         0.59277         0.35172               0.91597            0.81265           0.87485              
0.65057
   100         0.59543         0.35130               0.91311            0.81567           0.87394              
0.64984

## 8 · Validation

In [12]:
best_weights = TRAIN_DIR / "weights" / "best.pt"
print(f"Loading: {best_weights}")
model_eval = YOLO(str(best_weights))

print("Running validation...")
val_m = model_eval.val(
    data=str(YAML_PATH),
    split="val",
    imgsz=HP["imgsz"],
    batch=HP["batch"],
    device=DEVICE,
    verbose=True,
    plots=False,
)

print(f'\n{"="*40}')
print("VALIDATION RESULTS")
print(f"mAP@0.5      : {val_m.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {val_m.box.map:.4f}")
print(f"Precision    : {val_m.box.mp:.4f}")
print(f"Recall       : {val_m.box.mr:.4f}")
print(f'{"="*40}')

df_val_ap = pd.DataFrame(
    {"class": CLASS_NAMES, "AP@0.5": val_m.box.ap50, "AP@.5:.95": val_m.box.ap}
)
df_val_ap = df_val_ap.sort_values("AP@0.5", ascending=False)
print()
console.print(df_val_ap.to_string(index=False))

Loading: /kaggle/working/runs/detect/yolov8m_100e/weights/best.pt
Running validation...
Ultralytics 8.4.30 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,845,550 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 106.7±40.3 MB/s, size: 56.3 KB)
val: Scanning /kaggle/working/split_data/val/labels.cache... 280 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 280/280 73.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.2it/s 7.4s0.9ss
                   all        280       3554      0.922      0.813      0.878      0.654
               Hardhat        135        276      0.924      0.826      0.875      0.638
                  Mask        114        168      0.957      0.927      0.973       0.77
            NO-Hardhat        131        225 

class   AP@0.5  AP@.5:.95
          Mask 0.973380   0.770226
     machinery 0.965306   0.849889
        Person 0.939354   0.753763
NO-Safety Vest 0.913634   0.692130
    NO-Hardhat 0.878678   0.619403
       Hardhat 0.875137   0.638347
       vehicle 0.857049   0.639367
   Safety Vest 0.848570   0.642631
       NO-Mask 0.812656   0.520588
   Safety Cone 0.720411   0.416776

## 9 · Test Evaluation

In [13]:
print("Running test evaluation...")
test_m = model_eval.val(
    data=str(YAML_PATH),
    split="test",
    imgsz=HP["imgsz"],
    batch=HP["batch"],
    device=DEVICE,
    verbose=True,
    plots=False,
)

print(f'\n{"="*40}')
print("TEST RESULTS")
print(f"mAP@0.5      : {test_m.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {test_m.box.map:.4f}")
print(f"Precision    : {test_m.box.mp:.4f}")
print(f"Recall       : {test_m.box.mr:.4f}")
print(f'{"="*40}')

df_test_ap = pd.DataFrame(
    {"class": CLASS_NAMES, "AP@0.5": test_m.box.ap50, "AP@.5:.95": test_m.box.ap}
)
df_test_ap = df_test_ap.sort_values("AP@0.5", ascending=False)
print()
console.print(df_test_ap.to_string(index=False))

Running test evaluation...
Ultralytics 8.4.30 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7.4±2.1 MB/s, size: 51.6 KB)
val: Scanning /kaggle/working/split_data/test/labels... 281 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 281/281 295.4it/s 1.0s0.1s
val: New cache created: /kaggle/working/split_data/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.2it/s 7.8s0.9ss
                   all        281       4114      0.923      0.809      0.881      0.643
               Hardhat        139        391      0.932      0.747      0.828      0.577
                  Mask        120        186      0.945      0.922      0.955      0.743
            NO-Hardhat        152        259      0.927      0.792      0.876      0.615
               NO-Mask        158 

class   AP@0.5  AP@.5:.95
     machinery 0.964017   0.844765
          Mask 0.954821   0.743149
        Person 0.933838   0.735512
       vehicle 0.905668   0.722882
NO-Safety Vest 0.891666   0.669930
    NO-Hardhat 0.876234   0.614626
   Safety Vest 0.850516   0.620695
       NO-Mask 0.829819   0.488757
       Hardhat 0.827605   0.576675
   Safety Cone 0.772473   0.408097

## 10 · Export ONNX

In [14]:
print("Exporting to ONNX...")
onnx_path = model_eval.export(
    format="onnx",
    imgsz=HP["imgsz"],
    simplify=True,
    opset=17,
)
print(f"ONNX: {onnx_path}")

Exporting to ONNX...
Ultralytics 8.4.30 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/kaggle/working/runs/detect/yolov8m_100e/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 14, 8400) (49.6 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 249ms
Prepared 2 packages in 2.54s
Installed 2 packages in 11ms
 + onnxruntime-gpu==1.24.4
 + onnxslim==0.1.90

requirements: AutoUpdate success ✅ 3.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 17...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 6.4s, saved as '/kaggle/working/runs/detect/yolov8m_

## 11 · Zip & Download

In [15]:
ZIP_PATH = WORK_DIR / f"{RUN_NAME}.zip"

files = [
    (TRAIN_DIR / "weights" / "best.pt", "best.pt"),
    (TRAIN_DIR / "weights" / "last.pt", "last.pt"),
    (TRAIN_DIR / "results.csv", "results.csv"),
    (TRAIN_DIR / "args.yaml", "args.yaml"),
    (YAML_PATH, "ppe_data.yaml"),
]
if onnx_path and Path(onnx_path).exists():
    files.append((Path(onnx_path), "best.onnx"))

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for src, arc in files:
        if Path(src).exists():
            zf.write(src, arc)
        else:
            print(f"  skip (not found): {src}")

zip_mb = ZIP_PATH.stat().st_size / 1e6
print(f"ZIP ready: {ZIP_PATH.name}  ({zip_mb:.1f} MB)")
display(FileLink(str(ZIP_PATH), result_html_prefix="Download: "))

ZIP ready: yolov8m_100e.zip  (182.2 MB)


/kaggle/working/yolov8m_100e.zip

## 12 · Final Summary

In [16]:
total_time = time.time() - T0

print(f'\n{"="*60}')
print("FINAL SUMMARY")
print(f'{"="*60}')
print(f'Model              : YOLOv8{HP["model_size"]}')
print(f'Epochs trained     : {int(df_res["epoch"].iloc[-1])}  (best: {best_ep})')
print(f"Train / Val / Test : {len(train_idx)} / {len(val_idx)} / {len(test_idx)}")
print(f"Device             : {DEVICE}")
print(f"Total runtime      : {total_time/60:.1f} min")
print(f"Training time      : {train_time/60:.1f} min")
print()
print("Val  mAP@0.5       :", f"{val_m.box.map50:.4f}")
print("Val  mAP@0.5:0.95  :", f"{val_m.box.map:.4f}")
print("Val  Precision     :", f"{val_m.box.mp:.4f}")
print("Val  Recall        :", f"{val_m.box.mr:.4f}")
print()
print("Test mAP@0.5       :", f"{test_m.box.map50:.4f}")
print("Test mAP@0.5:0.95  :", f"{test_m.box.map:.4f}")
print("Test Precision     :", f"{test_m.box.mp:.4f}")
print("Test Recall        :", f"{test_m.box.mr:.4f}")
print(f'{"="*60}')

# Save summary JSON
summary = {
    "model": f"YOLOv8{HP['model_size']}",
    "epochs_trained": int(df_res["epoch"].iloc[-1]),
    "best_epoch": best_ep,
    "splits": {"train": len(train_idx), "val": len(val_idx), "test": len(test_idx)},
    "val": {
        "mAP50": round(val_m.box.map50, 4),
        "mAP5095": round(val_m.box.map, 4),
        "P": round(val_m.box.mp, 4),
        "R": round(val_m.box.mr, 4),
    },
    "test": {
        "mAP50": round(test_m.box.map50, 4),
        "mAP5095": round(test_m.box.map, 4),
        "P": round(test_m.box.mp, 4),
        "R": round(test_m.box.mr, 4),
    },
    "runtime_min": round(total_time / 60, 1),
}
with open(OUTPUT_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f'Summary JSON: {OUTPUT_DIR / "summary.json"}')


FINAL SUMMARY
Model              : YOLOv8m
Epochs trained     : 100  (best: 88)
Train / Val / Test : 2240 / 280 / 281
Device             : 0,1
Total runtime      : 72.7 min
Training time      : 72.0 min

Val  mAP@0.5       : 0.8784
Val  mAP@0.5:0.95  : 0.6543
Val  Precision     : 0.9222
Val  Recall        : 0.8129

Test mAP@0.5       : 0.8807
Test mAP@0.5:0.95  : 0.6425
Test Precision     : 0.9227
Test Recall        : 0.8092
Summary JSON: /kaggle/working/outputs/summary.json
